In [3]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F


In [4]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
def fetch_text_from_url(url):
    """Fetches and extracts main text from a webpage."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        res = requests.get(url, timeout=10, headers=headers)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        for element in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            element.decompose()

        text_parts = []

        content_selectors = [
            'main', 'article', '[role="main"]',
            '.content', '.main-content', '.post-content',
            '.entry-content', '.article-content', '.story-body'
        ]

        main_content = None
        for selector in content_selectors:
            main_content = soup.select_one(selector)
            if main_content:
                break

        if main_content:
            paragraphs = main_content.find_all(["p", "div", "span", "h1", "h2", "h3", "h4", "h5", "h6"])
            for p in paragraphs:
                text = p.get_text().strip()
                if text and len(text) > 20:  # Only include substantial text
                    text_parts.append(text)
        else:
            elements = soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6"])
            for element in elements:
                text = element.get_text().strip()
                if text and len(text) > 20:  # Only include substantial text
                    text_parts.append(text)

        if not text_parts:
            divs = soup.find_all("div")
            for div in divs:
                text = div.get_text().strip()
                if text and len(text.split()) > 10:  # At least 10 words
                    text_parts.append(text)

        full_text = " ".join(text_parts)

        import re
        full_text = re.sub(r'\s+', ' ', full_text)
        sentences = full_text.split('.')
        meaningful_sentences = [s.strip() for s in sentences if len(s.strip().split()) > 3]

        return '. '.join(meaningful_sentences).strip()

    except Exception as e:
        print(f"Failed to fetch from {url}: {e}")
        return ""


In [6]:
def simple_text_truncation(text, max_words=500):
    """Simple text truncation that preserves sentence boundaries."""
    words = text.split()
    if len(words) <= max_words:
        return text


    truncated = " ".join(words[:max_words])

    last_period = truncated.rfind('.')
    last_exclamation = truncated.rfind('!')
    last_question = truncated.rfind('?')

    last_sentence_end = max(last_period, last_exclamation, last_question)

    if last_sentence_end > len(truncated) * 0.7:
        return truncated[:last_sentence_end + 1]
    else:
        return truncated + "..."

In [7]:
def summarize_text(text, max_len=1000):
    if not text or not text.strip():
        return ""

    words = text.split()

    if len(words) <= max_len:
        return text

    if len(words) > 2000:
        text = simple_text_truncation(text, 1500)
        words = text.split()

    if len(words) > max_len:
        try:
            if len(words) < 50:
                return text

            input_length = len(words)
            min_length = max(30, min(100, input_length // 5))
            max_length = max(min_length + 50, min(512, input_length // 3))
            if max_length <= min_length:
                max_length = min_length + 50

            summary = summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False,
                clean_up_tokenization_spaces=True,
                no_repeat_ngram_size=3
            )

            return summary[0]['summary_text']

        except Exception as e:
            print(f"Summarization failed: {e}")
            return simple_text_truncation(text, max_len)

    return text


In [8]:
def compare_claim_with_evidence(claim, evidence):
    """Runs NLI model to compare claim and evidence."""
    if not evidence or not evidence.strip():
        return {
            "claim": claim,
            "label": "Wrong",
            "confidence": 0.0,
            "nli_label": "No Evidence"
        }

    processed_evidence = summarize_text(evidence)

    try:
        inputs = tokenizer.encode_plus(
            processed_evidence,
            claim,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)

            nli_labels = ["contradiction", "neutral", "entailment"]
            label_index = torch.argmax(probs).item()
            confidence = probs[0][label_index].item()
            nli_label = nli_labels[label_index]

            if nli_label == "entailment" and confidence > 0.75:
                final_label = "Strong"
            elif nli_label == "entailment" and confidence > 0.6:
                final_label = "Weak"
            elif nli_label == "neutral" and confidence > 0.7:
                final_label = "Weak"
            elif nli_label == "contradiction" and confidence > 0.7:
                final_label = "Wrong"
            else:
                final_label = "Weak"

            return {
                "claim": claim,
                "label": final_label,
                "confidence": round(confidence, 3),
                "nli_label": nli_label
            }

    except Exception as e:
        print(f"NLI comparison failed: {e}")
        return {
            "claim": claim,
            "label": "Wrong",
            "confidence": 0.0,
            "nli_label": "Error"
        }

In [41]:
def aggregate_results(results):
    """
    Aggregates multiple results with improved logic:
    - Considers both label frequency and confidence
    - Uses weighted scoring for final decision
    """
    if not results:
        return "Wrong", 0.0

    label_counts = {"Strong": 0, "Weak": 0, "Wrong": 0}
    confidences_by_label = {"Strong": [], "Weak": [], "Wrong": []}

    for res in results:
        label = res['label']
        confidence = res['confidence']
        label_counts[label] += 1
        confidences_by_label[label].append(confidence)

    weighted_scores = {}
    for label in ["Strong", "Weak", "Wrong"]:
        if confidences_by_label[label]:
            avg_conf = sum(confidences_by_label[label]) / len(confidences_by_label[label])
            weighted_scores[label] = label_counts[label] * avg_conf
        else:
            weighted_scores[label] = 0.0

    total_results = len(results)
    strong_ratio = label_counts["Strong"] / total_results
    wrong_ratio = label_counts["Wrong"] / total_results

    if strong_ratio >= 0.3 and weighted_scores["Strong"] > weighted_scores["Wrong"]:
        final_label = "Strong"
        avg_confidence = sum(confidences_by_label["Strong"]) / len(confidences_by_label["Strong"])
    elif wrong_ratio >= 0.6 and weighted_scores["Wrong"] > weighted_scores["Strong"]:
        final_label = "Wrong"
        avg_confidence = sum(confidences_by_label["Wrong"]) / len(confidences_by_label["Wrong"])
    else:
        final_label = "Weak"
        if confidences_by_label["Weak"]:
            avg_confidence = sum(confidences_by_label["Weak"]) / len(confidences_by_label["Weak"])
        else:
            all_confidences = [res['confidence'] for res in results]
            avg_confidence = sum(all_confidences) / len(all_confidences)

    return final_label, round(avg_confidence, 3)

In [10]:
def fact_check_claim_with_links(claim, urls, max_sources=10):
    """
    Fact-checks a claim against multiple URLs with improved error handling.
    """
    evidence_results = []
    processed_urls = 0

    for url in urls[:max_sources]:
        try:
            print(f"Processing: {url}")
            evidence_text = fetch_text_from_url(url)

            if evidence_text:
                result = compare_claim_with_evidence(claim, evidence_text)
                evidence_results.append(result)
                processed_urls += 1
            else:
                print(f"No text extracted from {url}")

        except Exception as e:
            print(f"Error processing {url}: {e}")

    if not evidence_results:
        return {
            "claim": claim,
            "final_label": "Wrong",
            "average_confidence": 0.0,
            "detailed_results": [],
            "sources_processed": 0,
            "error": "No evidence could be extracted from provided URLs"
        }

    final_label, avg_confidence = aggregate_results(evidence_results)

    return {
        "claim": claim,
        "final_label": final_label,
        "average_confidence": avg_confidence,
        "detailed_results": evidence_results,
        "sources_processed": processed_urls,
        "total_sources": len(urls)
    }

In [ ]:
if __name__ == "__main__":
    claim = "Bananas are vegetables."
    urls = [
        "https://www.fao.org/markets-and-trade/commodities-overview/bananas-tropical-fruits/bananas/en",
        "https://www.bananalink.org.uk/all-about-bananas/",
        "https://www.fairtrade.net/en/products/Fairtrade_products/Bananas.html",
        "https://www.britannica.com/plant/banana-plant",
        "https://nhb.gov.in/report_files/banana/BANANA.htm",
        "https://www.healthline.com/nutrition/foods/bananas",
        "https://www.webmd.com/food-recipes/health-benefits-bananas",
        "https://www.medanta.org/patient-education-blog/15-health-benefits-of-raw-bananas-and-why-you-should-eat-them",
        "https://www.livescience.com/45005-banana-nutrition-facts.html",
        "https://mydiagnostics.in/blogs/nutritional/the-best-banana-nutrition-health-benefits-and-facts-you-need-to-know",
        "https://www.growveg.com/plants/us-and-canada/how-to-grow-banana/"
    ]

    result = fact_check_claim_with_links(claim, urls)
    print("\n" + "="*50)
    print("FACT CHECK RESULT")
    print("="*50)
    print(f"Claim: {result['claim']}")
    print(f"Final Label: {result['final_label']}")
    print(f"Average Confidence: {result['average_confidence']}")
    print(f"Sources Processed: {result['sources_processed']}/{result['total_sources']}")
    print("\nDetailed Results:")
    for i, detail in enumerate(result['detailed_results'], 1):
        print(f"  {i}. Label: {detail['label']}, Confidence: {detail['confidence']}, NLI: {detail['nli_label']}")

Processing: https://www.fao.org/markets-and-trade/commodities-overview/bananas-tropical-fruits/bananas/en
Summarization failed: index out of range in self


Processing: https://www.bananalink.org.uk/all-about-bananas/
Summarization failed: index out of range in self
Processing: https://www.fairtrade.net/en/products/Fairtrade_products/Bananas.html
Summarization failed: index out of range in self
Processing: https://www.britannica.com/plant/banana-plant
Summarization failed: index out of range in self
Processing: https://nhb.gov.in/report_files/banana/BANANA.htm
Summarization failed: index out of range in self
Processing: https://www.healthline.com/nutrition/foods/bananas
Summarization failed: index out of range in self
Processing: https://www.webmd.com/food-recipes/health-benefits-bananas
Summarization failed: index out of range in self
Processing: https://www.medanta.org/patient-education-blog/15-health-benefits-of-raw-bananas-and-why-you-should-eat-them
Summarization failed: index out of range in self
Processing: https://www.livescience.com/45005-banana-nutrition-facts.html
Summarization failed: index out of range in self
Processing: http

In [12]:
pip install -U datasets


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 175.0 MB/s eta 0:00:0000:01
  Attempting uninstall: fsspec90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
    Found existing installation: fsspec 2025.3.2━━━━━━━━━━━━━━ 1/6 [pyarrow]
    Uninstalling fsspec-2025.3.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
      Successfully uninstalled fsspec-2025.3.2━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [datasets]5/6 [datasets]
Note: you may need to restart the kernel to use updated packages.


In [17]:
from datasets import load_dataset

fever_dataset = load_dataset("fever", "v1.0", split="labelled_dev" ,  trust_remote_code=True)

fever_dataset[0]


Generating train split:   0%|          | 0/311431 [00:00<?, ? examples/s]

Generating labelled_dev split:   0%|          | 0/37566 [00:00<?, ? examples/s]

Generating unlabelled_dev split:   0%|          | 0/19998 [00:00<?, ? examples/s]

Generating unlabelled_test split:   0%|          | 0/19998 [00:00<?, ? examples/s]

Generating paper_dev split:   0%|          | 0/18999 [00:00<?, ? examples/s]

Generating paper_test split:   0%|          | 0/18567 [00:00<?, ? examples/s]

{'id': 91198,
 'label': 'NOT ENOUGH INFO',
 'claim': 'Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.',
 'evidence_annotation_id': 108548,
 'evidence_id': -1,
 'evidence_wiki_url': '',
 'evidence_sentence_id': -1}

In [20]:
pip install wikipedia-api

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'wikipedia-api' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'wikipedia-api'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for wikipedia-api: filename=wikipedia_api-0.8.1-py3-none-any.whl size=15482 sha256=aac8f2c5306a5143b3cf423d9865e7885ec4df4a06d609bd895671c0d045fc08
  Stored in directory: /home/zeus/.cache/pip/wheels/1d/f8/07/0508c38722dcd82ee355e9d85e33c9e9471d4bec0f8ae72de0
Successfully built wikipedia-api
Note: you may need to restart the kernel to use updated packages.


In [22]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent='fact-check-bot/1.0 (contact@yourdomain.com)', language='en')

page = wiki.page("Colin_Kaepernick")
print(page.text[:500])  # print first 500 chars of the page text


Colin Rand Kaepernick ( KAP-ər-nik; born November 3, 1987) is an American civil rights activist and former professional football quarterback. He played six seasons for the San Francisco 49ers in the National Football League (NFL). In 2016, he gained national attention for kneeling during the national anthem at the start of NFL games in protest of police brutality and racial inequality in the United States.
Kaepernick played college football for the Nevada Wolf Pack, where he was named the Wester


In [23]:
def title_to_url(title):
    base_url = "https://en.wikipedia.org/wiki/"
    return base_url + title.replace(" ", "_")

In [36]:
# 1. Map FEVER labels to your NLI labels (adjust these mappings as needed)
nli_to_fever_label = {
    "entailment": "SUPPORTED",         # your NLI's label for supported claims
    "contradiction": "REFUTED",            # your NLI's label for refuted claims
    "neutral": "NOT ENOUGH INFO"      # your NLI's label for insufficient info
}


In [25]:
def extract_claim_and_evidence_titles(example):
    claim = example["claim"]
    
    # evidence is a list of evidence groups, each group is a list of evidence tuples
    # each evidence tuple has at least: [page_title, sentence_id, ...]
    evidence_titles = set()
    if "evidence" in example and example["evidence"] is not None:
        for ev_group in example["evidence"]:
            for ev in ev_group:
                if len(ev) > 0:
                    evidence_titles.add(ev[0])
    
    return claim, list(evidence_titles)

In [26]:
# 4. Run benchmarking for a limited number of samples
def run_fever_benchmark(nli_fact_check_fn, max_samples=100):
    fever_val = load_dataset("fever", "v1.0", split="labelled_dev" ,  trust_remote_code=True)
    
    total = 0
    correct = 0
    detailed_results = []
    
    for example in fever_val.select(range(max_samples)):
        claim, evidence_titles = extract_claim_and_evidence_titles(example)
        
        # Convert titles to URLs
        urls = [title_to_url(title) for title in evidence_titles]
        
        # Call your NLI fact-check function
        result = nli_fact_check_fn(claim, urls)
        
        # Map your NLI output label to FEVER label for evaluation
        pred_nli_label = result["final_label"]
        pred_fever_label = nli_to_fever_label.get(pred_nli_label, "NOT ENOUGH INFO")  # default fallback
        
        gold_label = example["label"]
        
        total += 1
        if pred_fever_label == gold_label:
            correct += 1
        
        detailed_results.append({
            "claim": claim,
            "gold_label": gold_label,
            "pred_label": pred_fever_label,
            "confidence": result.get("average_confidence", None),
        })
    
    accuracy = correct / total
    print(f"FEVER benchmark accuracy on {total} samples: {accuracy:.4f}")
    
    return detailed_results

# Example usage with your fact-check function
# detailed_results = run_fever_benchmark(fact_check_claim_with_links, max_samples=50)

In [40]:
detailed_results = run_fever_benchmark(fact_check_claim_with_links, max_samples=100)

FEVER benchmark accuracy on 100 samples: 0.1700


In [38]:
for r in detailed_results:
    print(f"\nClaim: {r['claim']}")
    print(f"True Label: {r['gold_label']}")
    print(f"Predicted: {r['pred_label']}")
    print(f"Confidence: {r['confidence']}")


Claim: Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.
True Label: NOT ENOUGH INFO
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Tilda Swinton is a vegan.
True Label: NOT ENOUGH INFO
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Fox 2000 Pictures released the film Soul Food.
True Label: SUPPORTS
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Fox 2000 Pictures released the film Soul Food.
True Label: SUPPORTS
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Fox 2000 Pictures released the film Soul Food.
True Label: SUPPORTS
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Fox 2000 Pictures released the film Soul Food.
True Label: SUPPORTS
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Fox 2000 Pictures released the film Soul Food.
True Label: SUPPORTS
Predicted: NOT ENOUGH INFO
Confidence: 0.0

Claim: Anne Rice was born in New Jersey.
True Label: NOT ENOUGH INFO
Predicted: NOT ENOUGH INFO
Confidence: 0.0